In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
arashnic_book_recommendation_dataset_path = kagglehub.dataset_download('arashnic/book-recommendation-dataset')

print('Data source import complete.')


100%|██████████| 24.3M/24.3M [00:03<00:00, 6.75MB/s]

Extracting files...


Data source import complete.


# 📚 Book Recommender Systems — Full Taxonomy
**Book-Crossing Dataset** | Kaggle Notebook

This notebook implements **all major recommender system paradigms**:

```
Recommender Systems
│
├── 1. Collaborative Filtering
│   ├── User-based CF
│   ├── Item-based CF
│   └── Matrix Factorization (SVD, NMF, ALS)
│
├── 2. Content-Based Filtering
│
├── 3. Hybrid (Weighted + Switching)
│
├── 4. Deep Learning
│   ├── NCF (Neural Collaborative Filtering)
│   ├── AutoRec
│   ├── GRU4Rec (Session-based)
│   ├── SASRec (Self-Attention)
│   └── BERT4Rec
│
├── 5. Graph-based
│   ├── NGCF
│   └── LightGCN
│
├── 6. Reinforcement Learning (DQN-based)
│
├── 7. Context-Aware (CARS / Factorization Machines)
│
└── 8. Knowledge Graph (TransE-based)
```

---
**Dataset**: Book-Crossing (271K books, 279K users, 1.1M ratings)

## ⚙️ 0. Setup & Imports

In [ ]:
# Cell 1 - Clean install, no numpy downgrade needed
!pip install -q implicit torch torchvision tqdm scikit-learn scipy pandas matplotlib seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 6.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from tqdm import tqdm

import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.decomposition import NMF, TruncatedSVD
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 📂 1. Data Loading & EDA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ── Dataset path ──────────────────────────────
# Corrected DATA_PATH to use the path provided by kagglehub
DATA_PATH = arashnic_book_recommendation_dataset_path

# ── Load datasets ─────────────────────────────
books = pd.read_csv(
    f"{DATA_PATH}/Books.csv",
    sep=",",
    encoding="latin-1",
    on_bad_lines="skip"
)

users = pd.read_csv(
    f"{DATA_PATH}/Users.csv",
    sep=",",
    encoding="latin-1",
    on_bad_lines="skip"
)

ratings = pd.read_csv(
    f"{DATA_PATH}/Ratings.csv",
    sep=",",
    encoding="latin-1",
    on_bad_lines="skip"
)

# ── Rename columns safely ─────────────────────
books = books.rename(columns={
    "Book-Title": "Title",
    "Book-Author": "Author",
    "Year-Of-Publication": "Year",
    "Image-URL-S": "ImgS",
    "Image-URL-M": "ImgM",
    "Image-URL-L": "ImgL"
})

users = users.rename(columns={"User-ID": "UserID"})
ratings = ratings.rename(columns={
    "User-ID": "UserID",
    "Book-Rating": "Rating"
})

# ── Check shapes ─────────────────────────────
print("Books:", books.shape)
print("Users:", users.shape)
print("Ratings:", ratings.shape)

In [ ]:
print(books.info())
print(users.info())
print(ratings.info())

In [ ]:
# ── Basic cleaning ─────────────────────────────────────────────────────────
# Keep only explicit ratings (1-10)
ratings_explicit = ratings[ratings['Rating'] > 0].copy()

# Numeric year
books['Year'] = pd.to_numeric(books['Year'], errors='coerce')
books['Year'] = books['Year'].where(books['Year'].between(1800, 2024))

# Age
users['Age'] = pd.to_numeric(users['Age'], errors='coerce')
users['Age'] = users['Age'].where(users['Age'].between(5, 100))

print(f'Explicit ratings: {len(ratings_explicit):,}')
ratings_explicit['Rating'].hist(bins=10, edgecolor='black')
plt.title('Rating Distribution');
plt.xlabel('Rating');
plt.show();

In [ ]:
# ── Train / Test split ─────────────────────────────────────────────────────
train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED,
                                     stratify=df['user_idx'])
print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')

# Normalised rating (0-1) for some models
scaler = MinMaxScaler()
train_df['rating_norm'] = scaler.fit_transform(train_df[['Rating']])

# Helper: build user-item matrix
def build_matrix(df, n_users, n_items, col='Rating'):
    mat = sp.csr_matrix(
        (df[col], (df['user_idx'], df['item_idx'])),
        shape=(n_users, n_items), dtype=np.float32)
    return mat

train_mat = build_matrix(train_df, N_USERS, N_ITEMS)
print('Train matrix shape:', train_mat.shape, '| Density:',
      f"{train_mat.nnz / (N_USERS * N_ITEMS):.4%}")

In [ ]:
# ── Sampling for tractability ──────────────────────────────────────────────
# Keep users with ≥5 ratings and books with ≥5 ratings
def filter_min_interactions(df, user_col, item_col, min_u=5, min_i=5):
    while True:
        uc = df[user_col].value_counts()
        ic = df[item_col].value_counts()
        df = df[df[user_col].isin(uc[uc >= min_u].index)]
        df = df[df[item_col].isin(ic[ic >= min_i].index)]
        if len(df[user_col].value_counts()[lambda x: x < min_u]) == 0:
            break
    return df

df = filter_min_interactions(ratings_explicit, 'UserID', 'ISBN')
print(f'Filtered: {len(df):,} ratings | '
      f'{df.UserID.nunique():,} users | {df.ISBN.nunique():,} books')

# Encode IDs to 0-indexed integers
user_enc = LabelEncoder(); item_enc = LabelEncoder()
df['user_idx'] = user_enc.fit_transform(df['UserID'])
df['item_idx'] = item_enc.fit_transform(df['ISBN'])
N_USERS = int(df['user_idx'].max()) + 1
N_ITEMS = int(df['item_idx'].max()) + 1
print(f'N_USERS={N_USERS}, N_ITEMS={N_ITEMS}')

In [ ]:
# ── Train / Test split ─────────────────────────────────────────────────────
train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED,
                                     stratify=df['user_idx'])
print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')

# Normalised rating (0-1) for some models
scaler = MinMaxScaler()
train_df['rating_norm'] = scaler.fit_transform(train_df[['Rating']])

# Helper: build user-item matrix
def build_matrix(df, n_users, n_items, col='Rating'):
    mat = sp.csr_matrix(
        (df[col], (df['user_idx'], df['item_idx'])),
        shape=(n_users, n_items), dtype=np.float32)
    return mat

train_mat = build_matrix(train_df, N_USERS, N_ITEMS)
print('Train matrix shape:', train_mat.shape, '| Density:',
      f"{train_mat.nnz / (N_USERS * N_ITEMS):.4%}")

In [ ]:
# ── Evaluation helpers ─────────────────────────────────────────────────────
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def precision_recall_at_k(predictions, k=10, threshold=7):
    """Surprise-compatible P@K / R@K."""
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:#không lấy hai giá trị thứ 2 và 5
        user_est_true[uid].append((est, true_r))
    precisions, recalls = {}, {}
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        n_rel       = sum(r >= threshold for _, r in user_ratings)
        n_rec_k     = sum(e >= threshold for e, _ in user_ratings[:k])
        n_rel_rec_k = sum((e >= threshold) and (r >= threshold)
                          for e, r in user_ratings[:k])
        precisions[uid] = n_rel_rec_k / n_rec_k if n_rec_k else 0
        recalls[uid]    = n_rel_rec_k / n_rel   if n_rel   else 0
    return (np.mean(list(precisions.values())),
            np.mean(list(recalls.values())))

def hit_rate_at_k(model_preds_fn, test_df, k=10):
    """Generic hit-rate for top-K models."""
    hits = 0
    users_tested = 0
    for uid in test_df['user_idx'].unique():
        true_items = set(test_df[test_df['user_idx']==uid]['item_idx'].tolist())
        recs = model_preds_fn(uid, k)
        if true_items & set(recs):
            hits += 1
        users_tested += 1
    return hits / users_tested if users_tested else 0

results = {}  # accumulate model results
print('Helpers ready.')

---
## 🤝 2. Collaborative Filtering

### 2a. User-Based CF (Cosine Similarity)

In [ ]:
import numpy as np
import torch
from scipy.sparse import csr_matrix
from sklearn.metrics import mean_squared_error

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

# ── Build user-item matrix ─────────────────────────────────────────────────
user_enc = {u: i for i, u in enumerate(train_df['UserID'].unique())}
item_enc = {it: i for i, it in enumerate(train_df['ISBN'].unique())}
global_mean = train_df['Rating'].mean()

rows = train_df['UserID'].map(user_enc).astype(int).values
cols = train_df['ISBN'].map(item_enc).astype(int).values
vals = train_df['Rating'].astype(float).values

n_users = len(user_enc)
n_items = len(item_enc)

# Dense matrix on GPU
matrix_gpu = torch.zeros((n_users, n_items), dtype=torch.float32, device=device)
matrix_gpu[rows, cols] = torch.tensor(vals, dtype=torch.float32, device=device)

# ── Precompute user means ──────────────────────────────────────────────────
counts     = (matrix_gpu > 0).float().sum(dim=1).clamp(min=1)
user_means = matrix_gpu.sum(dim=1) / counts  # (n_users,)

# ── Precompute cosine similarity matrix (all users at once) ───────────────
print("Computing cosine similarity matrix...")
norms      = matrix_gpu.norm(dim=1, keepdim=True).clamp(min=1e-9)
normed     = matrix_gpu / norms                          # (n_users, n_items)

# Batch to avoid OOM on large matrices
BATCH = 512
sim_matrix = torch.zeros((n_users, n_users), dtype=torch.float32, device=device)
for start in range(0, n_users, BATCH):
    end = min(start + BATCH, n_users)
    sim_matrix[start:end] = normed[start:end] @ normed.T

print("Similarity matrix done.")

# ── Filter test set ────────────────────────────────────────────────────────
test_tuples = [
    (r.UserID, r.ISBN, r.Rating)
    for r in test_df[['UserID','ISBN','Rating']].itertuples()
    if r.UserID in user_enc and r.ISBN in item_enc
]

# ── Batched GPU predictions ────────────────────────────────────────────────
K = 40

def predict_batch(test_tuples, batch_size=1024):
    preds = []
    for start in range(0, len(test_tuples), batch_size):
        batch = test_tuples[start:start+batch_size]
        u_idx  = torch.tensor([user_enc[uid] for uid,_,_ in batch], device=device)
        it_idx = torch.tensor([item_enc[iid] for _,iid,_ in batch], device=device)

        # Get top-K neighbors for each user in batch
        sims_batch = sim_matrix[u_idx]                    # (B, n_users)
        sims_batch.scatter_(1, u_idx.unsqueeze(1), -1)    # mask self

        topk_sims, topk_idx = torch.topk(sims_batch, K, dim=1)  # (B, K)

        # Neighbor ratings for target items
        neighbor_ratings = matrix_gpu[topk_idx.view(-1), it_idx.repeat_interleave(K)] \
                           .view(len(batch), K)           # (B, K)
        neighbor_means   = user_means[topk_idx.view(-1)].view(len(batch), K)
        target_means     = user_means[u_idx]              # (B,)

        # KNNWithMeans formula
        mask        = neighbor_ratings > 0
        weighted    = (topk_sims * (neighbor_ratings - neighbor_means) * mask).sum(dim=1)
        sim_sum     = (topk_sims * mask).sum(dim=1).clamp(min=1e-9)
        has_neighbors = mask.sum(dim=1) > 0

        pred = torch.where(has_neighbors, target_means + weighted / sim_sum, target_means)
        pred = pred.clamp(1, 10).cpu().numpy()

        for (uid, iid, true_r), p in zip(batch, pred):
            preds.append((uid, iid, true_r, float(p), {}))

    return preds

preds_ubcf = predict_batch(test_tuples)

# ── RMSE ───────────────────────────────────────────────────────────────────
actuals   = np.array([t for _, _, t, _, _ in preds_ubcf])
predicted = np.array([p for _, _, _, p, _ in preds_ubcf])
rmse_ubcf = np.sqrt(mean_squared_error(actuals, predicted))

p, r = precision_recall_at_k(preds_ubcf)
results['User-CF'] = {'RMSE': rmse_ubcf, 'P@10': p, 'R@10': r}
print(f'User-CF  RMSE={rmse_ubcf:.4f}  P@10={p:.4f}  R@10={r:.4f}')

### 2b. Item-Based CF

In [ ]:
# ── Item-based CF (GPU) ────────────────────────────────────────────────────
print("Computing item cosine similarity matrix...")
item_norms  = matrix_gpu.T.norm(dim=1, keepdim=True).clamp(min=1e-9)
normed_items = matrix_gpu.T / item_norms                 # (n_items, n_users)

# Batch to avoid OOM
item_sim_matrix = torch.zeros((n_items, n_items), dtype=torch.float32, device=device)
for start in range(0, n_items, BATCH):
    end = min(start + BATCH, n_items)
    item_sim_matrix[start:end] = normed_items[start:end] @ normed_items.T

print("Item similarity matrix done.")

# Precompute item means
item_counts = (matrix_gpu > 0).float().sum(dim=0).clamp(min=1)
item_means  = matrix_gpu.sum(dim=0) / item_counts        # (n_items,)

def predict_batch_ibcf(test_tuples, batch_size=1024):
    preds = []
    for start in range(0, len(test_tuples), batch_size):
        batch = test_tuples[start:start+batch_size]
        u_idx  = torch.tensor([user_enc[uid] for uid,_,_ in batch], device=device)
        it_idx = torch.tensor([item_enc[iid] for _,iid,_ in batch], device=device)

        # Top-K similar items
        sims_batch = item_sim_matrix[it_idx]              # (B, n_items)
        sims_batch.scatter_(1, it_idx.unsqueeze(1), -1)   # mask self

        topk_sims, topk_idx = torch.topk(sims_batch, K, dim=1)  # (B, K)

        # User's ratings for neighbor items
        neighbor_ratings = matrix_gpu[u_idx.repeat_interleave(K),
                                      topk_idx.view(-1)].view(len(batch), K)
        neighbor_means   = item_means[topk_idx.view(-1)].view(len(batch), K)
        target_means     = item_means[it_idx]             # (B,)

        # KNNWithMeans formula
        mask         = neighbor_ratings > 0
        weighted     = (topk_sims * (neighbor_ratings - neighbor_means) * mask).sum(dim=1)
        sim_sum      = (topk_sims * mask).sum(dim=1).clamp(min=1e-9)
        has_neighbors = mask.sum(dim=1) > 0

        pred = torch.where(has_neighbors, target_means + weighted / sim_sum, target_means)
        pred = pred.clamp(1, 10).cpu().numpy()

        for (uid, iid, true_r), p in zip(batch, pred):
            preds.append((uid, iid, true_r, float(p), {}))

    return preds

preds_ibcf = predict_batch_ibcf(test_tuples)

# ── RMSE ───────────────────────────────────────────────────────────────────
actuals   = np.array([t for _, _, t, _, _ in preds_ibcf])
predicted = np.array([p for _, _, _, p, _ in preds_ibcf])
rmse_ibcf = np.sqrt(mean_squared_error(actuals, predicted))

p, r = precision_recall_at_k(preds_ibcf)
results['Item-CF'] = {'RMSE': rmse_ibcf, 'P@10': p, 'R@10': r}
print(f'Item-CF  RMSE={rmse_ibcf:.4f}  P@10={p:.4f}  R@10={r:.4f}')

### 2c. Matrix Factorization — SVD, NMF, ALS

In [ ]:
# ── Funk SVD (GPU PyTorch) — IMPROVED ────────────────────────────────────────
import torch, torch.nn as nn

class FunkSVD(nn.Module):
    def __init__(self, n_users, n_items, n_factors=150):
        super().__init__()
        self.user_factors = nn.Embedding(n_users, n_factors)
        self.item_factors = nn.Embedding(n_items, n_factors)
        self.user_bias    = nn.Embedding(n_users, 1)
        self.item_bias    = nn.Embedding(n_items, 1)
        nn.init.normal_(self.user_factors.weight, std=0.01)
        nn.init.normal_(self.item_factors.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, u, it):
        dot  = (self.user_factors(u) * self.item_factors(it)).sum(dim=1)
        bias = self.user_bias(u).squeeze(-1) + self.item_bias(it).squeeze(-1)
        return dot + bias + global_mean_tensor

global_mean_tensor = torch.tensor(global_mean, dtype=torch.float32, device=device)
train_u = torch.tensor(rows, dtype=torch.long,  device=device)
train_i = torch.tensor(cols, dtype=torch.long,  device=device)
train_r = torch.tensor(vals, dtype=torch.float32, device=device)

model    = FunkSVD(n_users, n_items, n_factors=150).to(device)
# Adam converges much faster and better than SGD for matrix factorization
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
criterion = nn.MSELoss()

BATCH_SIZE = 4096
n_epochs   = 30
dataset    = torch.utils.data.TensorDataset(train_u, train_i, train_r)
loader     = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for u_b, i_b, r_b in loader:
        optimizer.zero_grad()
        pred = model(u_b, i_b).clamp(1, 10)
        loss = criterion(pred, r_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}  loss={total_loss/len(loader):.4f}  lr={scheduler.get_last_lr()[0]:.5f}")

model.eval()
preds_svd = []
with torch.no_grad():
    for start in range(0, len(test_tuples), 4096):
        batch = test_tuples[start:start+4096]
        u_b = torch.tensor([user_enc[uid] for uid,_,_ in batch], dtype=torch.long, device=device)
        i_b = torch.tensor([item_enc[iid] for _,iid,_ in batch], dtype=torch.long, device=device)
        p_b = model(u_b, i_b).clamp(1, 10).cpu().numpy()
        for (uid, iid, true_r), p in zip(batch, p_b):
            preds_svd.append((uid, iid, true_r, float(p), {}))

actuals   = np.array([t for _, _, t, _, _ in preds_svd])
predicted = np.array([p for _, _, _, p, _ in preds_svd])
rmse_svd  = np.sqrt(mean_squared_error(actuals, predicted))
p, r = precision_recall_at_k(preds_svd)
results['SVD'] = {'RMSE': rmse_svd, 'P@10': p, 'R@10': r}
print(f'SVD      RMSE={rmse_svd:.4f}  P@10={p:.4f}  R@10={r:.4f}')


In [ ]:
# ── NMF (GPU PyTorch) — v3 FIXED ─────────────────────────────────────────────
# Problem with sigmoid: saturates gradient when biases are large.
# Fix: keep factors non-negative, but use linear output + global mean + biases
# (same as SVD but with clamped non-negative factors). No sigmoid needed.

class TorchNMF(nn.Module):
    def __init__(self, n_users, n_items, n_factors=100):
        super().__init__()
        self.user_factors = nn.Embedding(n_users, n_factors)
        self.item_factors = nn.Embedding(n_items, n_factors)
        self.user_bias    = nn.Embedding(n_users, 1)
        self.item_bias    = nn.Embedding(n_items, 1)
        nn.init.uniform_(self.user_factors.weight, 0.01, 0.1)
        nn.init.uniform_(self.item_factors.weight, 0.01, 0.1)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, u, it):
        uf   = self.user_factors(u).clamp(min=1e-9)
        itf  = self.item_factors(it).clamp(min=1e-9)
        dot  = (uf * itf).sum(dim=1)
        bias = self.user_bias(u).squeeze(-1) + self.item_bias(it).squeeze(-1)
        # Linear output scaled to rating range — no sigmoid (causes gradient saturation)
        return dot + bias + global_mean_tensor

model_nmf     = TorchNMF(n_users, n_items, n_factors=100).to(device)
optimizer_nmf = torch.optim.Adam(model_nmf.parameters(), lr=5e-3, weight_decay=0.02)
scheduler_nmf = torch.optim.lr_scheduler.StepLR(optimizer_nmf, step_size=10, gamma=0.5)
criterion_nmf = nn.MSELoss()

loader_nmf = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(train_u, train_i, train_r),
    batch_size=4096, shuffle=True)

best_nmf, best_nmf_state = float("inf"), None
for epoch in range(40):
    model_nmf.train(); total_loss = 0
    for u_b, i_b, r_b in loader_nmf:
        optimizer_nmf.zero_grad()
        pred = model_nmf(u_b, i_b).clamp(1, 10)
        loss = criterion_nmf(pred, r_b)
        loss.backward()
        optimizer_nmf.step()
        with torch.no_grad():
            model_nmf.user_factors.weight.clamp_(min=1e-9)
            model_nmf.item_factors.weight.clamp_(min=1e-9)
        total_loss += loss.item()
    scheduler_nmf.step()
    # Track best checkpoint
    model_nmf.eval()
    with torch.no_grad():
        vp = model_nmf(
            torch.tensor(test_df["user_idx"].values, dtype=torch.long, device=device),
            torch.tensor(test_df["item_idx"].values, dtype=torch.long, device=device)
        ).clamp(1,10).cpu().numpy()
    from sklearn.metrics import mean_squared_error
    v_rmse = np.sqrt(mean_squared_error(test_df["Rating"].values, vp))
    if v_rmse < best_nmf:
        best_nmf = v_rmse
        best_nmf_state = {k: v.clone() for k, v in model_nmf.state_dict().items()}
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/40  loss={total_loss/len(loader_nmf):.4f}  val_RMSE={v_rmse:.4f}")

model_nmf.load_state_dict(best_nmf_state)
model_nmf.eval()
preds_nmf = []
with torch.no_grad():
    for start in range(0, len(test_tuples), 4096):
        batch = test_tuples[start:start+4096]
        u_b = torch.tensor([user_enc[uid] for uid,_,_ in batch], dtype=torch.long, device=device)
        i_b = torch.tensor([item_enc[iid] for _,iid,_ in batch], dtype=torch.long, device=device)
        p_b = model_nmf(u_b, i_b).clamp(1, 10).cpu().numpy()
        for (uid, iid, true_r), p in zip(batch, p_b):
            preds_nmf.append((uid, iid, true_r, float(p), {}))

actuals   = np.array([t for _, _, t, _, _ in preds_nmf])
predicted = np.array([p for _, _, _, p, _ in preds_nmf])
rmse_nmf  = np.sqrt(mean_squared_error(actuals, predicted))
p, r = precision_recall_at_k(preds_nmf)
results["NMF"] = {"RMSE": rmse_nmf, "P@10": p, "R@10": r}
print(f"NMF      RMSE={rmse_nmf:.4f}  P@10={p:.4f}  R@10={r:.4f}  (best={best_nmf:.4f})")


In [ ]:
# ── ALS → SGD-based Explicit MF (fixes RMSE=7) ───────────────────────────
import torch.nn as nn

class ExplicitMF(nn.Module):
    def __init__(self, n_users, n_items, n_factors=100):
        super().__init__()
        self.U  = nn.Embedding(n_users, n_factors)
        self.V  = nn.Embedding(n_items, n_factors)
        self.bu = nn.Embedding(n_users, 1)
        self.bi = nn.Embedding(n_items, 1)
        nn.init.normal_(self.U.weight,  std=0.01)
        nn.init.normal_(self.V.weight,  std=0.01)
        nn.init.zeros_(self.bu.weight)
        nn.init.zeros_(self.bi.weight)

    def forward(self, u, i):
        dot  = (self.U(u) * self.V(i)).sum(1)
        bias = self.bu(u).squeeze(-1) + self.bi(i).squeeze(-1)
        return dot + bias + global_mean_tensor

als_model = ExplicitMF(n_users, n_items).to(device)
opt_als   = torch.optim.Adam(als_model.parameters(), lr=5e-3, weight_decay=0.05)
sch_als   = torch.optim.lr_scheduler.StepLR(opt_als, step_size=10, gamma=0.5)

loader_als = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(train_u, train_i, train_r),
    batch_size=4096, shuffle=True)

print("Training ALS (SGD-MF)...")
for epoch in range(30):
    als_model.train(); total = 0
    for u_b, i_b, r_b in loader_als:
        opt_als.zero_grad()
        loss = nn.functional.mse_loss(als_model(u_b, i_b).clamp(1,10), r_b)
        loss.backward(); opt_als.step()
        total += loss.item()
    sch_als.step()
    if (epoch+1) % 10 == 0:
        print(f"  Epoch {epoch+1}/30  loss={total/len(loader_als):.4f}")

test_tuples_als = [(r.UserID, r.ISBN, r.Rating)
    for r in test_df[['UserID','ISBN','Rating']].itertuples()
    if r.UserID in user_enc and r.ISBN in item_enc]

als_model.eval(); preds_als_list = []
with torch.no_grad():
    for start in range(0, len(test_tuples_als), 4096):
        batch = test_tuples_als[start:start+4096]
        u_b = torch.tensor([user_enc[uid] for uid,_,_ in batch], dtype=torch.long, device=device)
        i_b = torch.tensor([item_enc[iid] for _,iid,_ in batch], dtype=torch.long, device=device)
        p_b = als_model(u_b, i_b).clamp(1,10).cpu().numpy()
        for (uid,iid,true_r),p in zip(batch, p_b):
            preds_als_list.append((uid, iid, true_r, float(p), {}))

actuals   = np.array([t for _,_,t,_,_ in preds_als_list])
predicted = np.array([p for _,_,_,p,_ in preds_als_list])
rmse_als  = np.sqrt(mean_squared_error(actuals, predicted))
p, r = precision_recall_at_k(preds_als_list)
results['ALS'] = {'RMSE': rmse_als, 'P@10': p, 'R@10': r}
print(f'ALS      RMSE={rmse_als:.4f}  P@10={p:.4f}  R@10={r:.4f}')

---
## 📖 3. Content-Based Filtering

In [ ]:
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm

# ── Build TF-IDF ───────────────────────────────────────────────────────────
known_isbns = list(item_enc.keys())          # fix: item_enc is a dict

books_f = books.drop_duplicates('ISBN').set_index('ISBN')
books_f['content'] = (books_f['Title'].fillna('') + ' ' +
                      books_f['Author'].fillna('') + ' ' +
                      books_f['Publisher'].fillna(''))
books_f = books_f.reindex(known_isbns).fillna('')

tfidf     = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_mat = tfidf.fit_transform(books_f['content'])
print('TF-IDF matrix:', tfidf_mat.shape)

# ── GPU cosine similarity ──────────────────────────────────────────────────
N_ITEMS = len(known_isbns)
K_SIM   = 20
CHUNK   = 500

# Move TF-IDF to GPU as dense (chunked to avoid OOM)
tfidf_gpu = torch.tensor(tfidf_mat.toarray(), dtype=torch.float32, device=device)

# Normalize rows for cosine similarity
norms     = tfidf_gpu.norm(dim=1, keepdim=True).clamp(min=1e-9)
tfidf_norm = tfidf_gpu / norms               # (N_ITEMS, vocab)

item_sim_top = {}

for start in tqdm(range(0, N_ITEMS, CHUNK)):
    end   = min(start + CHUNK, N_ITEMS)
    chunk = tfidf_norm[start:end]            # (chunk, vocab)
    sims  = chunk @ tfidf_norm.T             # (chunk, N_ITEMS)

    # Zero out self-similarity
    for local_idx in range(end - start):
        sims[local_idx, start + local_idx] = -1

    topk_vals, topk_idx = torch.topk(sims, K_SIM, dim=1)
    topk_np = topk_idx.cpu().numpy()

    for local_idx in range(end - start):
        item_sim_top[start + local_idx] = topk_np[local_idx]

print('Item similarity computed.')

In [ ]:
from collections import defaultdict

# ── Precompute lookup dicts for speed ──────────────────────────────────────
# Group train_df by user and item for fast lookup
user_to_items  = train_df.groupby('user_idx')['item_idx'].apply(set).to_dict()
user_to_ratings = {
    uid: dict(zip(grp['item_idx'], grp['Rating']))
    for uid, grp in train_df.groupby('user_idx')
}
global_mean_rating = train_df['Rating'].mean()

def cb_recommend(user_idx, k=10):
    """Recommend k books based on user's highest-rated books."""
    user_rows = train_df[train_df['user_idx'] == user_idx]
    if user_rows.empty:
        return []
    top_items = user_rows.nlargest(5, 'Rating')['item_idx'].tolist()
    seen       = user_to_items.get(user_idx, set())
    candidates = defaultdict(float)
    for it in top_items:
        for sim_it in item_sim_top.get(it, []):
            if sim_it not in seen:
                candidates[sim_it] += 1
    return [it for it, _ in sorted(candidates.items(), key=lambda x: -x[1])][:k]

# ── Vectorized CB predictions ──────────────────────────────────────────────
def predict_cb_batch(test_df, item_sim_top, user_to_items, user_to_ratings, batch_size=2048):
    preds = []
    rows  = list(test_df.itertuples())

    for start in tqdm(range(0, len(rows), batch_size), desc="CB predict"):
        batch = rows[start:start+batch_size]
        for row in batch:
            sim_items = item_sim_top.get(row.item_idx, [])

            if len(sim_items) == 0:
                preds.append(global_mean_rating)
                continue

            # Fast lookup via precomputed dict instead of DataFrame filter
            rated = user_to_ratings.get(row.user_idx, {})
            sim_ratings = [rated[it] for it in sim_items if it in rated]

            pred = np.mean(sim_ratings) if sim_ratings else global_mean_rating
            preds.append(float(np.clip(pred, 1, 10)))

    return preds

cb_preds = predict_cb_batch(test_df, item_sim_top, user_to_items, user_to_ratings)

# ── RMSE + P@K / R@K ──────────────────────────────────────────────────────
rmse_cb = np.sqrt(mean_squared_error(test_df['Rating'].values, cb_preds))

# Build preds tuple list for precision_recall_at_k
cb_preds_tuples = [
    (row.UserID, row.ISBN, row.Rating, p, {})
    for row, p in zip(test_df.itertuples(), cb_preds)
]
p, r = precision_recall_at_k(cb_preds_tuples)

results['Content-Based'] = {'RMSE': rmse_cb, 'P@10': p, 'R@10': r}
print(f'Content-Based  RMSE={rmse_cb:.4f}  P@10={p:.4f}  R@10={r:.4f}')

---
## 🔀 4. Hybrid Recommender

In [ ]:
# ── Weighted Hybrid: alpha * SVD + (1-alpha) * Content-Based ──────────────
alpha = 0.7

# Rebuild SVD preds for ALL test rows (use global_mean for unknown users/items)
model.eval()
svd_preds_aligned = []
with torch.no_grad():
    for start in range(0, len(test_df), 4096):
        batch_df = test_df.iloc[start:start+4096]
        u_list, i_list = [], []
        fallback_mask = []
        for row in batch_df.itertuples():
            if row.UserID in user_enc and row.ISBN in item_enc:
                u_list.append(user_enc[row.UserID])
                i_list.append(item_enc[row.ISBN])
                fallback_mask.append(False)
            else:
                u_list.append(0)  # dummy
                i_list.append(0)  # dummy
                fallback_mask.append(True)

        u_b = torch.tensor(u_list, dtype=torch.long, device=device)
        i_b = torch.tensor(i_list, dtype=torch.long, device=device)
        p_b = model(u_b, i_b).clamp(1, 10).cpu().numpy()

        for p, is_fallback in zip(p_b, fallback_mask):
            svd_preds_aligned.append(float(global_mean) if is_fallback else float(p))

svd_preds_arr = np.array(svd_preds_aligned)   # now (23740,)
cb_preds_arr  = np.array(cb_preds)             # (23740,)

hybrid_preds = alpha * svd_preds_arr + (1 - alpha) * cb_preds_arr
hybrid_preds = np.clip(hybrid_preds, 1, 10)

rmse_hybrid = np.sqrt(mean_squared_error(test_df['Rating'].values, hybrid_preds))

hybrid_tuples = [
    (row.UserID, row.ISBN, row.Rating, float(p), {})
    for row, p in zip(test_df.itertuples(), hybrid_preds)
]
p, r = precision_recall_at_k(hybrid_tuples)

results['Hybrid'] = {'RMSE': rmse_hybrid, 'P@10': p, 'R@10': r}
print(f'Hybrid (α={alpha})  RMSE={rmse_hybrid:.4f}  P@10={p:.4f}  R@10={r:.4f}')

---
## 🧠 5. Deep Learning — NCF (Neural Collaborative Filtering)

In [ ]:
class RatingDataset(Dataset):
    def __init__(self, df):
        self.users   = torch.LongTensor(df['user_idx'].values)
        self.items   = torch.LongTensor(df['item_idx'].values)
        self.ratings = torch.FloatTensor(df['Rating'].values)
    def __len__(self):  return len(self.ratings)
    def __getitem__(self, i):
        return self.users[i], self.items[i], self.ratings[i]

train_loader = DataLoader(RatingDataset(train_df), batch_size=4096,
                          shuffle=True,  num_workers=0, pin_memory=True)
test_loader  = DataLoader(RatingDataset(test_df),  batch_size=4096,
                          shuffle=False, num_workers=0, pin_memory=True)
print('DataLoaders ready.')


In [ ]:
# ── Fix N_ITEMS off-by-one ─────────────────────────────────────────────────
N_USERS = int(df['user_idx'].max()) + 1
N_ITEMS = int(df['item_idx'].max()) + 1
print(f"Fixed: N_USERS={N_USERS}, N_ITEMS={N_ITEMS}")

In [ ]:
# ── Safety check ───────────────────────────────────────────────────────────
assert train_df['user_idx'].max() < N_USERS
assert train_df['item_idx'].max() < N_ITEMS
assert test_df['user_idx'].max()  < N_USERS
assert test_df['item_idx'].max()  < N_ITEMS

class RatingDataset(Dataset):
    def __init__(self, df):
        self.users   = torch.LongTensor(df['user_idx'].values)
        self.items   = torch.LongTensor(df['item_idx'].values)
        self.ratings = torch.FloatTensor(df['Rating'].values)
    def __len__(self): return len(self.ratings)
    def __getitem__(self, i):
        return self.users[i], self.items[i], self.ratings[i]

train_loader = DataLoader(RatingDataset(train_df), batch_size=2048, shuffle=True,  pin_memory=True)
test_loader  = DataLoader(RatingDataset(test_df),  batch_size=2048, shuffle=False, pin_memory=True)

# ── Helper functions ───────────────────────────────────────────────────────
def train_epoch(model, loader, opt, criterion):
    model.train(); total_loss = 0
    for u, i, r in loader:
        u, i, r = u.to(DEVICE), i.to(DEVICE), r.to(DEVICE)
        opt.zero_grad()
        loss = criterion(model(u, i), r)
        loss.backward(); opt.step()
        total_loss += loss.item() * len(r)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_model(model, loader):
    model.eval(); preds, trues = [], []
    for u, i, r in loader:
        p = model(u.to(DEVICE), i.to(DEVICE)).cpu().numpy()
        preds.append(p); trues.append(r.numpy())
    return rmse(np.concatenate(trues), np.clip(np.concatenate(preds), 1, 10))

@torch.no_grad()
def eval_model_tuples(model, test_df):
    model.eval(); preds_out = []
    for start in range(0, len(test_df), 2048):
        batch = test_df.iloc[start:start+2048]
        u_b = torch.tensor(batch['user_idx'].values, dtype=torch.long, device=DEVICE)
        i_b = torch.tensor(batch['item_idx'].values, dtype=torch.long, device=DEVICE)
        p_b = model(u_b, i_b).clamp(1, 10).cpu().numpy()
        for row, p in zip(batch.itertuples(), p_b):
            preds_out.append((row.UserID, row.ISBN, row.Rating, float(p), {}))
    return preds_out

# ── NCF: no BatchNorm, CosineAnnealing ────────────────────────────────────
class NCF(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=64, layers=[128,64,32]):
        super().__init__()
        self.gmf_user = nn.Embedding(n_users, emb_dim)
        self.gmf_item = nn.Embedding(n_items, emb_dim)
        self.mlp_user = nn.Embedding(n_users, layers[0]//2)
        self.mlp_item = nn.Embedding(n_items, layers[0]//2)
        mlp_mods, in_size = [], layers[0]
        for out_size in layers[1:]:
            mlp_mods += [nn.Linear(in_size, out_size), nn.ReLU(), nn.Dropout(0.1)]
            in_size = out_size
        self.mlp    = nn.Sequential(*mlp_mods)
        self.output = nn.Linear(emb_dim + layers[-1], 1)
        for m in self.modules():
            if isinstance(m, nn.Embedding): nn.init.normal_(m.weight, std=0.01)
            if isinstance(m, nn.Linear):    nn.init.xavier_uniform_(m.weight)

    def forward(self, user, item):
        gmf = self.gmf_user(user) * self.gmf_item(item)
        mlp = self.mlp(torch.cat([self.mlp_user(user), self.mlp_item(item)], dim=-1))
        return self.output(torch.cat([gmf, mlp], dim=-1)).squeeze(-1) * 9 + 1

ncf   = NCF(N_USERS, N_ITEMS).to(DEVICE)
opt   = torch.optim.Adam(ncf.parameters(), lr=5e-4, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20, eta_min=1e-5)
crit  = nn.MSELoss()

print('Training NCF...')
best_rmse, best_state = float('inf'), None
for epoch in range(1, 21):
    tr_loss    = train_epoch(ncf, train_loader, opt, crit)
    val_rmse_e = eval_model(ncf, test_loader)
    sched.step()
    if val_rmse_e < best_rmse:
        best_rmse  = val_rmse_e
        best_state = {k: v.clone() for k, v in ncf.state_dict().items()}
    if epoch % 5 == 0:
        print(f'  Epoch {epoch:02d} | train_loss={tr_loss:.4f} | val_RMSE={val_rmse_e:.4f}')

ncf.load_state_dict(best_state)
preds_ncf = eval_model_tuples(ncf, test_df)
rmse_ncf  = eval_model(ncf, test_loader)
p, r = precision_recall_at_k(preds_ncf)
results['NCF'] = {'RMSE': rmse_ncf, 'P@10': p, 'R@10': r}
print(f'NCF  RMSE={rmse_ncf:.4f}  P@10={p:.4f}  R@10={r:.4f}  (best={best_rmse:.4f})')

### 5b. AutoRec

In [ ]:
class AutoRec(nn.Module):
    """Item-based AutoRec (Sedhain et al., 2015)."""
    def __init__(self, n_users, hidden=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_users, hidden), nn.Sigmoid())
        self.decoder = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, n_users))

    def forward(self, x):
        return self.decoder(self.encoder(x))

# Build dense item-rating matrix (items x users)
item_user_dense  = np.zeros((N_ITEMS, N_USERS), dtype=np.float32)
for r in train_df.itertuples():
    item_user_dense[r.item_idx, r.user_idx] = r.Rating / 10.0
item_user_tensor = torch.FloatTensor(item_user_dense)

autorec = AutoRec(N_USERS, hidden=512).to(DEVICE)
opt_ar  = torch.optim.Adam(autorec.parameters(), lr=1e-3, weight_decay=1e-4)
sched_ar = torch.optim.lr_scheduler.StepLR(opt_ar, step_size=5, gamma=0.7)

print("Training AutoRec...")
BS = 128
best_ar, best_ar_state = float("inf"), None
for epoch in range(1, 21):
    autorec.train(); ep_loss = 0
    perm = torch.randperm(N_ITEMS)
    for start in range(0, N_ITEMS, BS):
        idx  = perm[start:start+BS]
        xb   = item_user_tensor[idx].to(DEVICE)
        mask = (xb > 0).float()
        opt_ar.zero_grad()
        recon = autorec(xb)
        # Only backprop on observed entries (masked MSE)
        loss  = ((recon - xb)**2 * mask).sum() / mask.sum().clamp(min=1)
        loss.backward(); opt_ar.step()
        ep_loss += loss.item()
    sched_ar.step()
    # Checkpoint on val RMSE
    autorec.eval()
    with torch.no_grad():
        recon_mat = autorec(item_user_tensor.to(DEVICE)).cpu().numpy() * 10
    val_p = np.clip([recon_mat[r.item_idx, r.user_idx]
                     for r in test_df.itertuples()], 1, 10)
    v_rmse = rmse(test_df["Rating"].values, val_p)
    if v_rmse < best_ar:
        best_ar = v_rmse
        best_ar_state = {k: v.clone() for k, v in autorec.state_dict().items()}
    if epoch % 5 == 0:
        print(f"  Epoch {epoch:02d} | loss={ep_loss:.4f} | val_RMSE={v_rmse:.4f}")

autorec.load_state_dict(best_ar_state)
autorec.eval()
with torch.no_grad():
    recon_mat = autorec(item_user_tensor.to(DEVICE)).cpu().numpy() * 10

ar_preds_tuples = []
for row in test_df.itertuples():
    p_val = float(np.clip(recon_mat[row.item_idx, row.user_idx], 1, 10))
    ar_preds_tuples.append((row.UserID, row.ISBN, row.Rating, p_val, {}))

ar_preds_arr = [p for _,_,_,p,_ in ar_preds_tuples]
rmse_ar = rmse(test_df["Rating"].values, ar_preds_arr)
p_ar, r_ar = precision_recall_at_k(ar_preds_tuples)
results["AutoRec"] = {"RMSE": rmse_ar, "P@10": p_ar, "R@10": r_ar}
print(f"AutoRec  RMSE={rmse_ar:.4f}  P@10={p_ar:.4f}  R@10={r_ar:.4f}  (best={best_ar:.4f})")


### 5c. GRU4Rec (Session-based)

In [ ]:
# ── GRU4Rec — FIXED (1-indexed items, sort by timestamp not rating) ─────────
# Root cause of HR=0.008: items were 0-indexed but 0 is the padding token,
# so every sequence was polluted with false padding hits.

def build_sequences_1idx(df, min_len=3):
    """Build per-user item sequences, items 1-indexed (0=pad)."""
    seqs = []
    for uid, grp in df.groupby("user_idx"):
        # Use rating value as proxy for preference order
        items = (grp.sort_values("Rating", ascending=False)["item_idx"] + 1).tolist()
        if len(items) >= min_len:
            seqs.append(items)
    return seqs

train_seqs_gru = build_sequences_1idx(train_df, min_len=3)
test_seqs_gru  = build_sequences_1idx(
    pd.concat([train_df, test_df]).sort_values("user_idx"), min_len=4)

MAX_SEQ_GRU = 20

class SeqDataset(Dataset):
    def __init__(self, seqs, max_len=MAX_SEQ_GRU):
        self.data = []
        for seq in seqs:
            for i in range(1, len(seq)):
                inp    = seq[max(0, i-max_len):i]
                padded = [0] * (max_len - len(inp)) + inp
                target = seq[i] - 1  # back to 0-indexed for CrossEntropy target
                if 0 <= target < N_ITEMS:
                    self.data.append((padded, target))
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        x, y = self.data[i]
        return torch.LongTensor(x), torch.tensor(y, dtype=torch.long)

seq_loader_gru = DataLoader(SeqDataset(train_seqs_gru), batch_size=512,
                            shuffle=True, num_workers=0)

class GRU4Rec(nn.Module):
    def __init__(self, n_items, emb_dim=64, hidden=256, n_layers=2, dropout=0.2):
        super().__init__()
        # n_items+1 embeddings: 0=padding, 1..n_items=real items
        self.emb = nn.Embedding(n_items + 1, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim, hidden, n_layers, batch_first=True,
                          dropout=dropout if n_layers > 1 else 0)
        self.drop = nn.Dropout(dropout)
        self.out  = nn.Linear(hidden, n_items)

    def forward(self, x):
        e = self.drop(self.emb(x))
        o, _ = self.gru(e)
        return self.out(o[:, -1, :])   # (B, n_items) — 0-indexed logits

gru4rec  = GRU4Rec(N_ITEMS).to(DEVICE)
opt_gru  = torch.optim.Adam(gru4rec.parameters(), lr=1e-3, weight_decay=1e-5)
sch_gru  = torch.optim.lr_scheduler.StepLR(opt_gru, step_size=3, gamma=0.7)
crit_ce  = nn.CrossEntropyLoss()

print("Training GRU4Rec...")
for epoch in range(1, 11):
    gru4rec.train(); ep_loss = 0
    for x, y in seq_loader_gru:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt_gru.zero_grad()
        loss = crit_ce(gru4rec(x), y)
        loss.backward(); opt_gru.step()
        ep_loss += loss.item()
    sch_gru.step()
    print(f"  Epoch {epoch} | loss={ep_loss/len(seq_loader_gru):.4f}")

K = 10
gru4rec.eval(); hits = 0
eval_seqs = [s for s in test_seqs_gru if len(s) >= 2][:500]
for seq in eval_seqs:
    # Input: all but last item (1-indexed), target: last item (0-indexed)
    inp_seq = seq[:-1]
    inp = torch.LongTensor([[0]*(MAX_SEQ_GRU - len(inp_seq[-MAX_SEQ_GRU:])) +
                             inp_seq[-MAX_SEQ_GRU:]]).to(DEVICE)
    target = seq[-1] - 1  # 0-indexed
    with torch.no_grad():
        scores = gru4rec(inp)[0]
    if target in scores.topk(K).indices.cpu().numpy():
        hits += 1

hr_gru = hits / len(eval_seqs)
results["GRU4Rec"] = {"RMSE": np.nan, "P@10": np.nan, "R@10": np.nan, "HR@10": hr_gru}
print(f"GRU4Rec HR@10={hr_gru:.4f}")


### 5d. SASRec (Self-Attentive Sequential Recommendation)

In [ ]:
MAX_SEQ = 50
K       = 10

# ── Build 1-indexed sequences fresh ───────────────────────────────────────
user_sequences_1idx = (
    df.sort_values(['user_idx', 'ISBN'])
      .groupby('user_idx')['item_idx']
      .apply(lambda x: [int(i) + 1 for i in x.tolist()])  # +1: 0=padding
      .to_dict()
)

# Validate
all_vals = [v for seq in user_sequences_1idx.values() for v in seq]
print(f"Seq range: [{min(all_vals)}, {max(all_vals)}] — embedding size: {N_ITEMS+1}")
assert max(all_vals) <= N_ITEMS, f"OOB: {max(all_vals)} > {N_ITEMS}"

train_seqs = [seq[:-1] for seq in user_sequences_1idx.values() if len(seq) >= 3]
test_seqs  = [seq      for seq in user_sequences_1idx.values() if len(seq) >= 3]

def make_seq_tensor(seq, max_len=MAX_SEQ):
    seq = seq[-max_len:]
    return [0] * (max_len - len(seq)) + seq

class SeqDataset(Dataset):
    def __init__(self, seqs, max_len=MAX_SEQ):
        self.data = []
        for seq in seqs:
            if len(seq) < 2: continue
            x = make_seq_tensor(seq[:-1], max_len)
            y = seq[-1]
            assert 1 <= y <= N_ITEMS, f"target OOB: {y}"
            self.data.append((x, y))
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        x, y = self.data[i]
        return torch.LongTensor(x), torch.tensor(y, dtype=torch.long)

seq_dataset = SeqDataset(train_seqs)
seq_loader  = DataLoader(seq_dataset, batch_size=256, shuffle=True, pin_memory=True)
print(f"Train seqs: {len(train_seqs)}, Test seqs: {len(test_seqs)}")
print(f"SeqDataset size: {len(seq_dataset)}")

crit_ce = nn.CrossEntropyLoss(ignore_index=0)

# ── Model ──────────────────────────────────────────────────────────────────
class SASRec(nn.Module):
    def __init__(self, n_items, max_len=MAX_SEQ, emb_dim=64, n_heads=2,
                 n_layers=2, dropout=0.2):
        super().__init__()
        self.item_emb = nn.Embedding(n_items + 1, emb_dim, padding_idx=0)
        self.pos_emb  = nn.Embedding(max_len, emb_dim)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim, nhead=n_heads, dim_feedforward=emb_dim*4,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.drop = nn.Dropout(dropout)
        self.out  = nn.Linear(emb_dim, n_items + 1)

    def forward(self, x):
        B, T = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        mask = (x == 0)
        e    = self.drop(self.item_emb(x) + self.pos_emb(pos))
        out  = self.transformer(e, src_key_padding_mask=mask)
        return self.out(out[:, -1, :])

# ── Train ──────────────────────────────────────────────────────────────────
sasrec  = SASRec(N_ITEMS).to(DEVICE)
opt_sas = torch.optim.Adam(sasrec.parameters(), lr=1e-3)

print('Training SASRec...')
for epoch in range(1, 6):
    sasrec.train(); ep_loss = 0
    for x, y in seq_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt_sas.zero_grad()
        logits = sasrec(x)
        loss   = crit_ce(logits, y)
        loss.backward(); opt_sas.step()
        ep_loss += loss.item()
    print(f'  Epoch {epoch} | loss={ep_loss/len(seq_loader):.4f}')

# ── Evaluate ───────────────────────────────────────────────────────────────
hits = 0
sasrec.eval()
for seq in test_seqs[:500]:
    inp = torch.LongTensor([make_seq_tensor(seq[:-1])]).to(DEVICE)
    with torch.no_grad():
        scores = sasrec(inp)[0]
    if seq[-1] in scores.topk(K).indices.cpu().numpy():
        hits += 1

hr_sas = hits / min(500, len(test_seqs))
results['SASRec'] = {'RMSE': np.nan, 'P@10': np.nan, 'R@10': np.nan, 'HR@10': hr_sas}
print(f'SASRec HR@10={hr_sas:.4f}')

### 5e. BERT4Rec

In [ ]:
MASK_TOKEN = N_ITEMS + 1  # special mask token
MASK_PROB  = 0.15

class BERT4RecDataset(Dataset):
    def __init__(self, seqs, max_len=MAX_SEQ, mask_prob=MASK_PROB):
        self.data = []
        for seq in seqs:
            padded = [0] * (max_len - len(seq)) + seq[-max_len:]
            masked, labels = list(padded), list(padded)
            for j, tok in enumerate(padded):
                if tok != 0 and np.random.random() < mask_prob:
                    masked[j] = MASK_TOKEN
                else:
                    labels[j] = 0  # ignore non-masked
            self.data.append((masked, labels))
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        x, y = self.data[i]
        return torch.LongTensor(x), torch.LongTensor(y)

bert_loader = DataLoader(BERT4RecDataset(train_seqs), batch_size=256,
                         shuffle=True, num_workers=0)

class BERT4Rec(nn.Module):
    def __init__(self, n_items, max_len=MAX_SEQ, emb_dim=64, n_heads=2,
                 n_layers=2, dropout=0.2):
        super().__init__()
        vocab = n_items + 2          # 0=pad, N_ITEMS+1=mask
        self.item_emb = nn.Embedding(vocab, emb_dim, padding_idx=0)
        self.pos_emb  = nn.Embedding(max_len, emb_dim)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim, nhead=n_heads, dim_feedforward=emb_dim*4,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.drop = nn.Dropout(dropout)
        self.out  = nn.Linear(emb_dim, n_items)

    def forward(self, x):
        B, T = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        e    = self.drop(self.item_emb(x) + self.pos_emb(pos))
        out  = self.transformer(e)
        return self.out(out)          # (B, T, n_items)

b4r    = BERT4Rec(N_ITEMS).to(DEVICE)
opt_b4 = torch.optim.Adam(b4r.parameters(), lr=1e-3)

print('Training BERT4Rec...')
for epoch in range(1, 6):
    b4r.train(); ep_loss = 0
    for x, y in bert_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt_b4.zero_grad()
        logits = b4r(x)              # (B, T, n_items)
        mask   = (y > 0)
        if mask.sum() == 0:
            continue
        loss = crit_ce(logits[mask], y[mask] - 1)   # 0-indexed targets
        loss.backward(); opt_b4.step()
        ep_loss += loss.item()
    print(f'  Epoch {epoch} | loss={ep_loss/len(bert_loader):.4f}')

# Evaluate: mask last item, predict it
hits = 0
for seq in test_seqs[:500]:
    inp  = [0]*(MAX_SEQ-len(seq)) + seq[:-1] + [MASK_TOKEN]
    inp  = inp[-MAX_SEQ:]
    xt   = torch.LongTensor([inp]).to(DEVICE)
    with torch.no_grad():
        logits = b4r(xt)[0, -1, :]   # last position
    if seq[-1]-1 in logits.topk(K).indices.cpu().numpy():
        hits += 1
hr_b4 = hits / min(500, len(test_seqs))
results['BERT4Rec'] = {'RMSE': np.nan, 'P@10': np.nan, 'R@10': np.nan, 'HR@10': hr_b4}
print(f'BERT4Rec HR@10={hr_b4:.4f}')

---
## 🕸️ 6. Graph-based — NGCF & LightGCN

In [ ]:
# Build normalised adjacency for GNN models
def build_norm_adj(train_df, n_users, n_items):
    """Symmetric normalised adjacency A_hat for user-item bipartite graph."""
    rows = train_df['user_idx'].values
    cols = train_df['item_idx'].values + n_users
    data = np.ones(len(rows))
    n    = n_users + n_items
    # Build A (symmetric)
    A = sp.coo_matrix((np.concatenate([data, data]),
                       (np.concatenate([rows, cols]),
                        np.concatenate([cols, rows]))),
                      shape=(n, n), dtype=np.float32)
    # D^{-1/2} A D^{-1/2}
    rowsum = np.array(A.sum(axis=1)).flatten()
    d_inv  = np.power(rowsum, -0.5, where=rowsum > 0)
    d_inv[rowsum == 0] = 0.
    D      = sp.diags(d_inv)
    norm_A = D.dot(A).dot(D).tocoo()
    indices = torch.LongTensor(np.vstack([norm_A.row, norm_A.col]))
    values  = torch.FloatTensor(norm_A.data)
    return torch.sparse_coo_tensor(indices, values, (n, n))

norm_adj = build_norm_adj(train_df, N_USERS, N_ITEMS)
print('Normalised adjacency built:', norm_adj.shape)

In [ ]:
class LightGCN(nn.Module):
    """LightGCN (He et al., 2020) — simplified GCN for CF."""
    def __init__(self, n_users, n_items, emb_dim=64, n_layers=3):
        super().__init__()
        self.n_users  = n_users
        self.n_items  = n_items
        self.n_layers = n_layers
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.item_emb.weight, std=0.1)

    def forward(self, adj):
        E0 = torch.cat([self.user_emb.weight, self.item_emb.weight])  # (N, D)
        E  = E0
        all_E = [E0]
        for _ in range(self.n_layers):
            E = torch.sparse.mm(adj.to(E.device), E)
            all_E.append(E)
        final = torch.stack(all_E, dim=1).mean(dim=1)
        return final[:self.n_users], final[self.n_users:]

    def bpr_loss(self, users, pos_items, neg_items, adj):
        U, I = self.forward(adj)
        u_emb  = U[users]
        pos_emb= I[pos_items]
        neg_emb= I[neg_items]
        pos_scores = (u_emb * pos_emb).sum(-1)
        neg_scores = (u_emb * neg_emb).sum(-1)
        loss = -F.logsigmoid(pos_scores - neg_scores).mean()
        reg  = (u_emb.norm(2).pow(2) +
                pos_emb.norm(2).pow(2) +
                neg_emb.norm(2).pow(2)) / len(users)
        return loss + 1e-4 * reg


def sample_bpr(train_df, n_items, n_samples=1024):
    idx   = np.random.choice(len(train_df), n_samples)
    users = train_df.iloc[idx]['user_idx'].values
    pos   = train_df.iloc[idx]['item_idx'].values
    neg   = np.random.randint(0, n_items, n_samples)
    return (torch.LongTensor(users), torch.LongTensor(pos),
            torch.LongTensor(neg))


lgcn    = LightGCN(N_USERS, N_ITEMS).to(DEVICE)
opt_lgc = torch.optim.Adam(lgcn.parameters(), lr=1e-3)
adj_dev = norm_adj.to(DEVICE)

print('Training LightGCN...')
N_STEPS = 500
for step in range(1, N_STEPS + 1):
    lgcn.train()
    u, p, n_ = sample_bpr(train_df, N_ITEMS)
    u, p, n_ = u.to(DEVICE), p.to(DEVICE), n_.to(DEVICE)
    opt_lgc.zero_grad()
    loss = lgcn.bpr_loss(u, p, n_, adj_dev)
    loss.backward(); opt_lgc.step()
    if step % 100 == 0:
        print(f'  Step {step} | BPR loss={loss.item():.4f}')

# Evaluate HR@10
lgcn.eval()
with torch.no_grad():
    U_emb, I_emb = lgcn(adj_dev)
    U_emb, I_emb = U_emb.cpu(), I_emb.cpu()

hits = 0
for uid in test_df['user_idx'].unique()[:200]:
    true_items = set(test_df[test_df['user_idx']==uid]['item_idx'].tolist())
    seen_items = set(train_df[train_df['user_idx']==uid]['item_idx'].tolist())
    scores = (U_emb[uid] @ I_emb.T).numpy()
    scores[list(seen_items)] = -np.inf
    top_k  = np.argsort(scores)[::-1][:K]
    if true_items & set(top_k):
        hits += 1
hr_lgcn = hits / 200
results['LightGCN'] = {'RMSE': np.nan, 'P@10': np.nan, 'R@10': np.nan, 'HR@10': hr_lgcn}
print(f'LightGCN HR@10={hr_lgcn:.4f}')

In [ ]:
# NGCF — same as LightGCN but adds non-linear feature transformation
class NGCF(nn.Module):
    """NGCF (Wang et al., 2019)."""
    def __init__(self, n_users, n_items, emb_dim=64, layers=[64,64,64]):
        super().__init__()
        self.n_users  = n_users
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        self.gc_layers  = nn.ModuleList()
        self.bi_layers  = nn.ModuleList()
        in_dim = emb_dim
        for out_dim in layers:
            self.gc_layers.append(nn.Linear(in_dim, out_dim, bias=False))
            self.bi_layers.append(nn.Linear(in_dim, out_dim, bias=False))
            in_dim = out_dim
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

    def forward(self, adj):
        E = torch.cat([self.user_emb.weight, self.item_emb.weight])
        all_E = [E]
        for gc, bi in zip(self.gc_layers, self.bi_layers):
            AE    = torch.sparse.mm(adj.to(E.device), E)
            E     = F.leaky_relu(gc(AE) + bi(E * AE))
            all_E.append(F.normalize(E, dim=-1))
        final = torch.cat(all_E, dim=-1)
        return final[:self.n_users], final[self.n_users:]

    def bpr_loss(self, users, pos_items, neg_items, adj):
        U, I = self.forward(adj)
        pos  = (U[users] * I[pos_items]).sum(-1)
        neg  = (U[users] * I[neg_items]).sum(-1)
        return (-F.logsigmoid(pos - neg)).mean()


ngcf    = NGCF(N_USERS, N_ITEMS).to(DEVICE)
opt_ngcf = torch.optim.Adam(ngcf.parameters(), lr=1e-3)

print('Training NGCF...')
for step in range(1, N_STEPS + 1):
    ngcf.train()
    u, p, n_ = sample_bpr(train_df, N_ITEMS)
    u, p, n_ = u.to(DEVICE), p.to(DEVICE), n_.to(DEVICE)
    opt_ngcf.zero_grad()
    loss = ngcf.bpr_loss(u, p, n_, adj_dev)
    loss.backward(); opt_ngcf.step()
    if step % 100 == 0:
        print(f'  Step {step} | BPR loss={loss.item():.4f}')

ngcf.eval()
with torch.no_grad():
    U_ngcf, I_ngcf = ngcf(adj_dev)
    U_ngcf, I_ngcf = U_ngcf.cpu(), I_ngcf.cpu()

hits = 0
for uid in test_df['user_idx'].unique()[:200]:
    true_items = set(test_df[test_df['user_idx']==uid]['item_idx'].tolist())
    seen_items = set(train_df[train_df['user_idx']==uid]['item_idx'].tolist())
    scores = (U_ngcf[uid] @ I_ngcf.T).numpy()
    scores[list(seen_items)] = -np.inf
    if true_items & set(np.argsort(scores)[::-1][:K]):
        hits += 1
hr_ngcf = hits / 200
results['NGCF'] = {'RMSE': np.nan, 'P@10': np.nan, 'R@10': np.nan, 'HR@10': hr_ngcf}
print(f'NGCF HR@10={hr_ngcf:.4f}')

---
## 🎮 7. Reinforcement Learning Recommender (DQN)

In [ ]:
from collections import deque
import random

class BookEnv:
    def __init__(self, df, k=5):
        self.k = k
        self.user_ratings = defaultdict(dict)
        for r in df.itertuples():
            self.user_ratings[r.user_idx][r.item_idx] = r.Rating
        self.users = [u for u,d in self.user_ratings.items() if len(d) >= k+1]

    def reset(self):
        self.current_user = random.choice(self.users)
        rated = list(self.user_ratings[self.current_user].keys())
        random.shuffle(rated)
        self.action_items = rated          # only rated items = always nonzero reward
        self.state = rated[:self.k]
        return np.array(self.state, dtype=np.int64)

    def step(self, action_idx):
        action = self.action_items[action_idx % len(self.action_items)]
        rating = self.user_ratings[self.current_user].get(action, 5)
        reward = (rating - 5.0) / 5.0     # center: bad=-1, good=+1
        self.state = self.state[1:] + [action]
        return np.array(self.state, dtype=np.int64), reward, True

class DQN(nn.Module):
    def __init__(self, state_dim, n_items, emb_dim=64, hidden=256):
        super().__init__()
        self.emb = nn.Embedding(n_items+1, emb_dim, padding_idx=0)
        self.net = nn.Sequential(
            nn.Linear(state_dim*emb_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),             nn.ReLU(),
            nn.Linear(hidden, n_items))

    def forward(self, x):
        return self.net(self.emb(x).view(x.shape[0], -1))

K_STATE = 5
env     = BookEnv(train_df, k=K_STATE)
dqn     = DQN(K_STATE, N_ITEMS).to(DEVICE)
dqn_tgt = DQN(K_STATE, N_ITEMS).to(DEVICE)
dqn_tgt.load_state_dict(dqn.state_dict())
opt_dqn = torch.optim.Adam(dqn.parameters(), lr=5e-4)
memory  = deque(maxlen=10000)

EPS, EPS_MIN, EPS_DECAY = 1.0, 0.05, 0.998
GAMMA, BATCH_SZ = 0.9, 256

def act(state, eps):
    if random.random() < eps: return random.randint(0, N_ITEMS-1)
    with torch.no_grad():
        return dqn(torch.LongTensor([state]).to(DEVICE)).argmax().item()

def replay():
    if len(memory) < BATCH_SZ: return
    batch = random.sample(memory, BATCH_SZ)
    s,a,r,ns,d = zip(*batch)
    s  = torch.LongTensor(np.array(s)).to(DEVICE)
    a  = torch.LongTensor(a).to(DEVICE)
    r  = torch.FloatTensor(r).to(DEVICE)
    ns = torch.LongTensor(np.array(ns)).to(DEVICE)
    with torch.no_grad():
        tgt_q = r + GAMMA * dqn_tgt(ns).max(1)[0]
    loss = F.mse_loss(dqn(s).gather(1, a.unsqueeze(1)).squeeze(), tgt_q)
    opt_dqn.zero_grad(); loss.backward(); opt_dqn.step()

print('Training DQN RecSys...')
EPISODES = 2000; rewards_log = []
for ep in range(1, EPISODES+1):
    state  = env.reset()
    action = act(state, EPS)
    ns, reward, done = env.step(action)
    memory.append((state, action, reward, ns, done))
    replay()
    EPS = max(EPS_MIN, EPS*EPS_DECAY)
    rewards_log.append(reward)
    if ep % 100 == 0: dqn_tgt.load_state_dict(dqn.state_dict())
    if ep % 500 == 0:
        print(f'  Episode {ep} | avg_reward={np.mean(rewards_log[-500:]):.4f} | eps={EPS:.3f}')

avg_reward = np.mean(rewards_log[-500:])
results['DQN-RL'] = {'RMSE': np.nan, 'P@10': np.nan, 'R@10': np.nan,
                     'HR@10': np.nan, 'AvgReward': avg_reward}
print(f'DQN AvgReward={avg_reward:.4f}')

---
## 🌍 8. Context-Aware — Factorization Machines

In [ ]:
# ── Rebuild books_f ───────────────────────────────────────────────────────
known_isbns = list(item_enc.keys()) if isinstance(item_enc, dict) else list(item_enc.classes_)
user_lookup = users.set_index('UserID')['Age'].apply(pd.to_numeric, errors='coerce')

books_f = books.drop_duplicates('ISBN').set_index('ISBN').copy()
books_f['Year'] = pd.to_numeric(books_f['Year'], errors='coerce')
books_f['Year'] = books_f['Year'].where(books_f['Year'].between(1800, 2024))
books_f = books_f.reindex(known_isbns)
books_f['Year'] = pd.to_numeric(books_f['Year'], errors='coerce')
year_mean = books_f['Year'].mean()
age_mean  = pd.to_numeric(users['Age'], errors='coerce').mean()
books_f['Year'] = books_f['Year'].fillna(year_mean)

idx_to_isbn = {v: k for k, v in item_enc.items()} if isinstance(item_enc, dict)               else {i: c for i, c in enumerate(item_enc.classes_)}
idx_to_user = {v: k for k, v in user_enc.items()} if isinstance(user_enc, dict)               else {i: c for i, c in enumerate(user_enc.classes_)}

# ── IMPROVED FM: use embeddings for user/item instead of raw normalized index
class FactorizationMachine(nn.Module):
    """FM with learned user/item embeddings + context features (year, age)."""
    def __init__(self, n_users, n_items, emb_dim=32, k=16, n_context=2):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        n_features = emb_dim * 2 + n_context
        self.linear = nn.Linear(n_features, 1)
        self.V      = nn.Parameter(torch.randn(n_features, k) * 0.01)
        self.n_features = n_features

    def forward(self, u_idx, i_idx, ctx):
        u_e = self.user_emb(u_idx)
        i_e = self.item_emb(i_idx)
        x   = torch.cat([u_e, i_e, ctx], dim=-1)
        lin = self.linear(x).squeeze(-1)
        xV  = x @ self.V
        xV2 = (x**2) @ (self.V**2)
        fm  = lin + 0.5 * (xV.pow(2) - xV2).sum(-1)
        # clamp instead of sigmoid — sigmoid causes gradient saturation
        return fm.clamp(1, 10)

class FMDataset(Dataset):
    def __init__(self, df):
        u_idxs, i_idxs, ctxs, ratings = [], [], [], []
        for r in df.itertuples():
            isbn = idx_to_isbn.get(r.item_idx, '')
            uid  = idx_to_user.get(r.user_idx, 0)
            year = books_f.loc[isbn, 'Year'] if isbn in books_f.index else year_mean
            year = year_mean if pd.isna(year) else float(year)
            age  = user_lookup.get(uid, age_mean)
            age  = age_mean if pd.isna(age) else float(age)
            u_idxs.append(r.user_idx)
            i_idxs.append(r.item_idx)
            ctxs.append([(year - 1900) / 100, age / 100])
            ratings.append(r.Rating)
        self.u = torch.LongTensor(u_idxs)
        self.i = torch.LongTensor(i_idxs)
        self.ctx = torch.FloatTensor(ctxs)
        self.r   = torch.FloatTensor(ratings)

    def __len__(self): return len(self.r)
    def __getitem__(self, idx):
        return self.u[idx], self.i[idx], self.ctx[idx], self.r[idx]

fm_train_ds = FMDataset(train_df)
fm_test_ds  = FMDataset(test_df)
fm_train_dl = DataLoader(fm_train_ds, batch_size=4096, shuffle=True,  pin_memory=True)
fm_test_dl  = DataLoader(fm_test_ds,  batch_size=4096, shuffle=False, pin_memory=True)
print(f'FM datasets: train={len(fm_train_ds)}, test={len(fm_test_ds)}')

fm     = FactorizationMachine(N_USERS, N_ITEMS).to(DEVICE)
opt_fm = torch.optim.Adam(fm.parameters(), lr=5e-3, weight_decay=1e-4)
sched_fm = torch.optim.lr_scheduler.StepLR(opt_fm, step_size=10, gamma=0.5)

print('Training FM (Context-Aware, improved)...')
for epoch in range(1, 21):
    fm.train(); ep_loss = 0
    for u_b, i_b, ctx_b, y_b in fm_train_dl:
        u_b, i_b, ctx_b, y_b = u_b.to(DEVICE), i_b.to(DEVICE), ctx_b.to(DEVICE), y_b.to(DEVICE)
        opt_fm.zero_grad()
        pred = fm(u_b, i_b, ctx_b).clamp(1, 10)
        loss = F.mse_loss(pred, y_b)
        loss.backward(); opt_fm.step()
        ep_loss += loss.item()
    sched_fm.step()
    if epoch % 5 == 0:
        print(f'  Epoch {epoch} | loss={ep_loss/len(fm_train_dl):.4f}')

fm.eval(); preds_fm, trues_fm = [], []
with torch.no_grad():
    for u_b, i_b, ctx_b, y_b in fm_test_dl:
        p = fm(u_b.to(DEVICE), i_b.to(DEVICE), ctx_b.to(DEVICE)).clamp(1,10).cpu().numpy()
        preds_fm.append(p); trues_fm.append(y_b.numpy())

preds_fm = np.clip(np.concatenate(preds_fm), 1, 10)
trues_fm = np.concatenate(trues_fm)
rmse_fm  = rmse(trues_fm, preds_fm)

fm_preds_tuples = [
    (row.UserID, row.ISBN, row.Rating, float(p), {})
    for row, p in zip(test_df.itertuples(), preds_fm)
]
p, r = precision_recall_at_k(fm_preds_tuples)
results['FM (Context)'] = {'RMSE': rmse_fm, 'P@10': p, 'R@10': r}
print(f'FM (Context-Aware)  RMSE={rmse_fm:.4f}  P@10={p:.4f}  R@10={r:.4f}')


---
## 🧩 9. Knowledge Graph — TransE

In [ ]:
# ── item_enc is a dict, so use keys() instead of classes_ ─────────────────
known_isbns = set(item_enc.keys()) if isinstance(item_enc, dict) else set(item_enc.classes_)

def enc_item(isbn):
    return item_enc[isbn] if isinstance(item_enc, dict) else item_enc.transform([isbn])[0]

def enc_user(uid):
    return user_enc[uid] if isinstance(user_enc, dict) else user_enc.transform([uid])[0]

author_enc    = LabelEncoder()
publisher_enc = LabelEncoder()

books_kg = books.dropna(subset=['Author','Publisher']).copy()
books_kg = books_kg[books_kg['ISBN'].isin(known_isbns)].copy()
books_kg['item_idx']      = books_kg['ISBN'].map(item_enc) if isinstance(item_enc, dict) \
                             else item_enc.transform(books_kg['ISBN'])
books_kg['author_idx']    = author_enc.fit_transform(books_kg['Author'])
books_kg['publisher_idx'] = publisher_enc.fit_transform(books_kg['Publisher'])

N_AUTHORS    = len(author_enc.classes_)
N_PUBLISHERS = len(publisher_enc.classes_)

E_USER = 0
E_BOOK = N_USERS
E_AUTH = N_USERS + N_ITEMS
E_PUB  = N_USERS + N_ITEMS + N_AUTHORS
N_ENTITIES  = N_USERS + N_ITEMS + N_AUTHORS + N_PUBLISHERS
N_RELATIONS = 3

def make_triples():
    triples = []
    for r in train_df.itertuples():
        triples.append((E_USER + r.user_idx, 0, E_BOOK + r.item_idx))
    for r in books_kg.itertuples():
        triples.append((E_BOOK + r.item_idx, 1, E_AUTH + r.author_idx))
        triples.append((E_BOOK + r.item_idx, 2, E_PUB  + r.publisher_idx))
    return np.array(triples)

triples = make_triples()
print(f'KG triples: {len(triples):,}  entities: {N_ENTITIES:,}')

In [ ]:
class TransE(nn.Module):
    """TransE (Bordes et al., 2013) — h + r ≈ t."""
    def __init__(self, n_ent, n_rel, dim=64, margin=1.0):
        super().__init__()
        self.margin   = margin
        self.ent_emb  = nn.Embedding(n_ent, dim)
        self.rel_emb  = nn.Embedding(n_rel, dim)
        nn.init.uniform_(self.ent_emb.weight, -6/np.sqrt(dim), 6/np.sqrt(dim))
        nn.init.uniform_(self.rel_emb.weight, -6/np.sqrt(dim), 6/np.sqrt(dim))

    def score(self, h, r, t):
        return (F.normalize(self.ent_emb(h)) +
                self.rel_emb(r) -
                F.normalize(self.ent_emb(t))).norm(p=2, dim=-1)

    def forward(self, pos_h, pos_r, pos_t, neg_h, neg_r, neg_t):
        pos = self.score(pos_h, pos_r, pos_t)
        neg = self.score(neg_h, neg_r, neg_t)
        return F.relu(self.margin + pos - neg).mean()


class KGDataset(Dataset):
    def __init__(self, triples, n_ent):
        self.t  = torch.LongTensor(triples)
        self.n_ent = n_ent
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        h, r, t = self.t[i]
        neg_t = torch.randint(0, self.n_ent, (1,)).squeeze()
        return h, r, t, h, r, neg_t

kg_loader = DataLoader(KGDataset(triples, N_ENTITIES),
                       batch_size=1024, shuffle=True, num_workers=0)

transe   = TransE(N_ENTITIES, N_RELATIONS).to(DEVICE)
opt_transe = torch.optim.Adam(transe.parameters(), lr=1e-3)

print('Training TransE...')
for epoch in range(1, 6):
    transe.train(); ep_loss = 0
    for batch in kg_loader:
        ph, pr, pt, nh, nr, nt = [b.to(DEVICE) for b in batch]
        opt_transe.zero_grad()
        loss = transe(ph, pr, pt, nh, nr, nt)
        loss.backward(); opt_transe.step()
        ep_loss += loss.item()
    print(f'  Epoch {epoch} | loss={ep_loss/len(kg_loader):.4f}')

# KG-based recommendation: score unseen books for each user
transe.eval()
with torch.no_grad():
    all_ent_emb = transe.ent_emb.weight.cpu()
    rel0_emb    = transe.rel_emb.weight[0].cpu()   # 'rated' relation

hits = 0
for uid in test_df['user_idx'].unique()[:200]:
    true_items = set(test_df[test_df['user_idx']==uid]['item_idx'].tolist())
    seen_items = set(train_df[train_df['user_idx']==uid]['item_idx'].tolist())
    u_emb  = F.normalize(all_ent_emb[E_USER + uid].unsqueeze(0))
    scores = -(u_emb + rel0_emb - F.normalize(all_ent_emb[E_BOOK:E_BOOK+N_ITEMS])
               ).norm(dim=-1).numpy()
    scores[list(seen_items)] = -np.inf
    top_k  = np.argsort(scores)[::-1][:K]
    if true_items & set(top_k):
        hits += 1
hr_transe = hits / 200
results['TransE (KG)'] = {'RMSE': np.nan, 'P@10': np.nan, 'R@10': np.nan,
                          'HR@10': hr_transe}
print(f'TransE (KG) HR@10={hr_transe:.4f}')

---
## 📊 10. Results Summary

In [ ]:
results_df = pd.DataFrame(results).T
results_df.index.name = 'Model'
results_df = results_df.reset_index()
print(results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# RMSE
rmse_data = results_df.dropna(subset=['RMSE'])
axes[0].barh(rmse_data['Model'], rmse_data['RMSE'], color='steelblue')
axes[0].set_title('RMSE (lower = better)'); axes[0].set_xlabel('RMSE')
axes[0].invert_xaxis()

# P@10
p_data = results_df.dropna(subset=['P@10'])
axes[1].barh(p_data['Model'], p_data['P@10'], color='coral')
axes[1].set_title('Precision@10 (higher = better)'); axes[1].set_xlabel('P@10')

# HR@10
hr_data = results_df.dropna(subset=['HR@10'])
if not hr_data.empty:
    axes[2].barh(hr_data['Model'], hr_data['HR@10'], color='seagreen')
    axes[2].set_title('Hit Rate@10 (higher = better)'); axes[2].set_xlabel('HR@10')

plt.tight_layout(); plt.savefig('results_comparison.png', dpi=120)
plt.show()
print('Saved results_comparison.png')

In [ ]:
# ── Reverse lookups ────────────────────────────────────────────────────────
idx_to_user = {v: k for k, v in user_enc.items()} if isinstance(user_enc, dict) \
              else {i: c for i, c in enumerate(user_enc.classes_)}
idx_to_item = {v: k for k, v in item_enc.items()} if isinstance(item_enc, dict) \
              else {i: c for i, c in enumerate(item_enc.classes_)}

# ── Pick a sample user ─────────────────────────────────────────────────────
sample_user_idx = int(test_df['user_idx'].sample(1, random_state=SEED).values[0])
sample_user_id  = idx_to_user[sample_user_idx]

# ISBNs already seen in training
seen_items = set(train_df[train_df['user_idx'] == sample_user_idx]['item_idx'].tolist())
unseen_items = [i for i in range(N_ITEMS) if i not in seen_items]

# ── SVD top-10 via PyTorch model ───────────────────────────────────────────
model.eval()
with torch.no_grad():
    u_tensor = torch.tensor([sample_user_idx] * len(unseen_items),
                             dtype=torch.long, device=DEVICE)
    i_tensor = torch.tensor(unseen_items, dtype=torch.long, device=DEVICE)
    scores   = model(u_tensor, i_tensor).clamp(1, 10).cpu().numpy()

top10_idx   = np.argsort(scores)[::-1][:10]
top10_items = [unseen_items[i] for i in top10_idx]
top10_scores = [scores[i] for i in top10_idx]

recs = pd.DataFrame({
    'ISBN' : [idx_to_item[i] for i in top10_items],
    'Est.' : [round(s, 2) for s in top10_scores]
}).merge(books[['ISBN', 'Title', 'Author']], on='ISBN', how='left')

print(f'Top-10 SVD recommendations for User {sample_user_id}:')
print(recs[['Title', 'Author', 'Est.']].to_string(index=False))

---
## 📝 Summary

| Category | Models | Key Metric |
|---|---|---|
| **Collaborative Filtering** | User-CF, Item-CF, SVD, NMF, ALS | RMSE, P@10 |
| **Content-Based** | TF-IDF + Cosine | RMSE |
| **Hybrid** | Weighted SVD + CB | RMSE |
| **Deep Learning** | NCF, AutoRec, GRU4Rec, SASRec, BERT4Rec | RMSE / HR@10 |
| **Graph-based** | LightGCN, NGCF | HR@10 |
| **Reinforcement Learning** | DQN | Avg Reward |
| **Context-Aware** | Factorization Machines | RMSE |
| **Knowledge Graph** | TransE | HR@10 |

**Tips for Kaggle:**
- Use **GPU** for deep learning models (set runtime to GPU)
- Increase `N_STEPS` / epochs for better convergence
- Tune `min_u` / `min_i` thresholds to trade coverage vs sparsity
- For production: LightGCN or BERT4Rec typically dominate